### Notebook for ANN implementation for Emotion Recognition
Before running this notebook parser.py would need to be run in order to generate the needed features.csv file, this file contains data from every user in the dataset and used every recommended method to create 513 features per user.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
# Needed to access the minilearn folder
sys.path.insert(0, str(Path.cwd().parent))

from minilearn.preprocessing import train_test_split


df = pd.read_csv('../features.csv')

X = df.iloc[:, 7:]
y = df['emotion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

First we demonstrate a simple example with a Dense ANN

In [2]:
from minilearn.classifiers import DenseANN
# Retreive dimension size of input features
_, dim = X_train.shape
d_ann = DenseANN(dim, [256, 256, 128], 8, epochs=20)
d_ann.fit(X_train, y_train)
d_ann_score = d_ann.score(X_test, y_test)
print(f"Dense ANN Accuracy: {d_ann_score:0.2f}")

Dense ANN Accuracy: 0.35


With this linear NN with 3 hidden layers we get a modest accuracy of 34% for classifying emotion.
Since this version runs relativley fast we will up the layers and epoch size, scaling this simple design up.

In [3]:
d_ann = DenseANN(dim, [512, 1024, 1024, 512, 256, 128], 8, epochs=100)
d_ann.fit(X_train, y_train)
d_ann_score = d_ann.score(X_test, y_test)
print(f"Large Dense ANN Accuracy: {d_ann_score:0.2f}")

Large Dense ANN Accuracy: 0.38


As we see with this test even with a much larger dense model and longer training time it struggles to converge.

This is likely due to the large amount of noise in the model from the 513 features, so we will first use feature selection to slim this down as well as use the standard scaler.

In [3]:
from minilearn.preprocessing import StandardScaler
from minilearn.dim_reduction import pca

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_reduced = pca(X_scaled, 128)
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

d_ann = DenseANN(128, [256, 256, 128], 8, epochs=20)
d_ann.fit(X_train, y_train)
d_ann_score = d_ann.score(X_test, y_test)
print(f"Dense ANN Accuracy: {d_ann_score:0.2f}")

Dense ANN Accuracy: 0.72


Reducing the features down to 128 and applying the standard scaler clearly made a massive difference, bringing the model accuracy up to 72%!

With this improvement and level of accuracy we likely have the right architecture at this point, so next we will start tuning the parameters using grid CV search in order to find the correct hyperparameters.

In [5]:
from minilearn.model_selection import GridSearchCV
import torch.nn as nn
import torch

params = {
    "input_dim": 128,
    "num_classes": 8,
    "learning_rate": [0.001, 0.01, 0.1],
    "epochs": [10, 20, 50],
    "optimizer": [torch.optim.Adam, torch.optim.AdamW, torch.optim.SGD],
    "hidden_layers": [[256, 256, 128], [256, 512, 256, 128], [128, 128, 128]]
}
gs = GridSearchCV(DenseANN, params)
gs.fit(X_train, y_train)
gs_score = gs.score(X_test, y_test)
print(f"Grid Search Best Score: {gs_score:0.2f}")
print(gs.best_params_)

Grid Search Best Score: 0.67
{'input_dim': 128, 'num_classes': 8, 'learning_rate': 0.01, 'epochs': 20, 'optimizer': <class 'torch.optim.adam.Adam'>, 'hidden_layers': [256, 256, 128]}


This strategy did not seem to pay off, actually producing a lower value than the prior model.

So we will instead pivot to general ANN's instead of just Dense neural networks.

In [6]:
from minilearn.classifiers import ANN

layers = [
    nn.Unflatten(dim=1, unflattened_size=(1, 128)),
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Conv1d(1, 4, 4, 1, 1),
    nn.ReLU(),
    nn.Conv1d(4, 8, 4, 1, 2),
    nn.Flatten(),
    nn.ReLU(),
    nn.Linear(256*8, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 8),
    nn.ReLU()
]
ann = ANN(layers, epochs=50)
ann.fit(X_train, y_train)
ann_score = ann.score(X_test, y_test)
print(f"ANN Accuracy: {ann_score:0.2f}")

ANN Accuracy: 0.38


Over several attempts we did not find a good ANN convolutional model for this task so we will go back to the original Dense model.

In [9]:
from minilearn.ensemble import VotingClassifier

d_ann = DenseANN(128, [256, 256, 128], 8, epochs=40)
voters = VotingClassifier(d_ann, 10)
voters.fit(X_train, y_train)
voters_score = voters.score(X_test, y_test)
print(f"Dense ANN Accuracy: {voters_score:0.2f}")

Dense ANN Accuracy: 0.05


In [ ]:
for est in voters.estimators:
    print(f"Accuracy: {est.score(X_test, y_test):0.2f}")

for est in voters.estimators:
    print(f"Accuracy Full: {est.score(X_train, y_train):0.2f}")

[2 4 3 1 2 3 0 2 4 2 6 6 2 2 3 2 2 4 5 2 2 4 0 1 3 4 2 6 5 7 4 5 3 7 5 4 3
 3 4 3 3 4 2 1 2 4 5 1 6 3 1 4 6 6 3 4 5 0 5 2 1 2 5 3 3 3 2 4 0 3 4 1 5 5
 1 2 4 0 4 3 6 3 3 5 2 4 7 4 3 5 1 0 4 2 4 1 1 2 4 7 5 1 5 1 3 4 2 4 2 7 2
 4 4 5 1 3 3 1 7 1 2 1 7 0 2 6 2 3 2 1 1 3 4 6 0 5 7 7 2 3 5 1 0 5 4 3 5 1
 2 4 1 6 4 3 7 4 0 5 2 4 4 1 4 5 1 1 6 0 5 1 5 2 1 5 1 4 5 7 3 1 1 1 6 2 3
 1 1 4 1 3 3 3 5 0 0 5 0 0 4 1 1 4 2 0 6 4 6 1 3 5 7 5 3 1 4 0 1 3 1 2 6 3
 5 3 5 2 1 3 2 4 3 6 5 5 2 2 4 5 3 5 0 1 1 5 3 3 5 2 5 5 5 5 2 4 4 4 3 2 5
 3 3 2 2 3 1 3 3 2 3 7 1 6 4 3 1 2 3 0 2 1 4 5 5 1 6 3 2 0 1 3 4 3 5 1 2 1
 0 1 1 4 2 0 3 1 1 3 5 1 1 4 2 3 2 1 6 1 5 2 5 5 1 5 3 1 5 2 7 1 4 4 2 7 2
 2 5 1 3 4 2 5 0 1 4 2 1 3 5 3 1 5 2 2 1 2 3 5 1 7 2 3 1 2 2 6 4 4 6 1 3 1
 2 1 2 2 3 3 3 3 5 1 2 4 1 4 1 0 2 2 4 2 2 4 5 4 4 4 1 2 7 3 2 7 7 4 0 4 3
 5 1 5 5 0 3 2 1 4 4 4 3 1 5 6 4 3 1 3 3 3 1 4 4 2 3 0 0 2 0 4 7 2 6 3 4 2
 6 2 3 0 1 1 1 7 2 0 3 1 4 0 2 3 4 4 5 6 6 6 4 2 2 3 5 3 1 4 6 6 0 3 1 6 6
 5 5 3 3 6 4 4 3 1]
[3 3 